[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Fancy Indexing


## What you will be able to do

Pick readings with a list of positions, in any order and with repeats, sort stations and days with
`np.argsort`, take a grid of rows and hours with `np.ix_`, and tell when the result is a copy.

## The idea

### The problem

A slice picks evenly spaced positions, a range of hours or every 24th one. Many of the positions a
question needs follow no such pattern. Noon on the first of each month comes 744, 672 or 720 hours
after the one before it, depending on how long the month was. The stations from coldest to warmest
are rows 2, 3, 1 and 0. Bergen's five warmest days are wherever they happened to fall in July.

With lists, each of these is a list comprehension, `[bergen[h] for h in noons]`, and the result is a
Python list again, without any of an array's methods. Sorting is harder still. `sorted` puts the
values in order and forgets where each one came from, so after sorting Bergen's daily means there is
no way left to say which day was the warmest.

### What fancy indexing is

> In NumPy, **fancy indexing** is indexing an array with a list or array of integer positions. The
> result holds the values at those positions, in the order the positions are given, with the shape of
> the index, and it is always a new array holding a copy of the values.

### Why it works that way

A list of positions is ordinary data, so it can come from a calculation. `hour_of_year` makes the
positions of the twelve noons. `np.argsort` returns the positions that would put an array's values
in order, so they sort the values and still say which day each value belongs to. `np.nanargmin`
gives the position of each station's coldest hour. Positions can repeat and arrive in any order,
because NumPy looks each one up separately.

Two lists of positions, one for each dimension, pair up. `readings[[0, 3], [12, 756]]` is two
readings, row 0 at position 12 and row 3 at position 756, and the grid of all four combinations
needs `np.ix_`, which turns the two lists into a selector for every row with every position.

Because the positions can come in any order, with repeats, NumPy cannot describe the result as a
regular step through the array's memory, which is what a view needs. So reading with a fancy index
makes a copy, and changing the copy leaves the array alone. Assignment works differently. With the
positions on the left of `=`, as in `readings[[0, 2], 0] = 0.0`, NumPy writes into the array
itself. Two of this notebook's Common errors come from confusing them.

### Where this shows up

Sorting a table by one of its columns follows the same steps as `np.argsort`, putting the row
positions in order by that column and then picking every column with them. The **Pandas, Deep Dive**
guide sorts with `sort_values` and picks rows by position with `.iloc`. A random sample of rows is a
fancy index whose positions were drawn at random, which the **Random Numbers** notebook draws.
Labeling values, such as the station name for each row a calculation picked, is `names[positions]`.

### What this notebook covers

- picking rows and hours with a list of positions, in any order and with repeats
- `np.argsort`, to sort stations and days and keep track of which was which
- two lists of positions that pair up, and `np.ix_` for a grid
- the copy a fancy index returns, checked with `np.shares_memory`
- assigning through a fancy index, which writes into the array

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

celsius = np.array([2.5, -1.0, 0.5, -3.0])
print(celsius[[3, 0, 0]])
order = np.argsort(celsius, kind="stable")
print(order)
print(celsius[order])
```

```
[-3.   2.5  2.5]
[3 1 2 0]
[-3.  -1.   0.5  2.5]
```

The list of positions picked the last reading and then the first one twice. `np.argsort` returned
the positions in the order that sorts the values, and indexing with them sorted the array.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### Rows in an order of your choosing

A list of row positions keeps whole rows, in the order of the list. `names` holds the station names
as an array, so the same positions label the rows they picked:

In [2]:
names = np.array(list(STATIONS))
pair = readings[[0, 2]]

print(pair.shape)
print(names[[0, 2]])
print(names[[3, 0]])
print(readings[[3, 0], :3])

(2, 8760)
['Bergen' 'Svalbard']
['Tromso' 'Bergen']
[[-6.7 -8.5 -8.5]
 [-3.6 -3.7 -3.7]]


`readings[[3, 0], :3]` put Tromso's first three hours above Bergen's, the reverse of their order in
the array. A list of positions and a slice can index the same array, one for each dimension.

### Positions along the hours

The positions can come from a calculation. Here a list comprehension asks `hour_of_year` for noon on
the first of every month:

In [3]:
noons = [hour_of_year(month, 1, 12) for month in range(1, 13)]

print(noons)
print(readings[0, noons])
print(readings[0, [noons[6], noons[6], noons[0]]])

[12, 756, 1428, 2172, 2892, 3636, 4356, 5100, 5844, 6564, 7308, 8028]
[ 0.9  1.3  2.7  7.2 11.9 16.6 19.1 18.1 16.3 12.5  8.3  4.3]
[19.1 19.1  0.9]


The gaps between the noons are 744, 672 and 720 hours, which no slice's step can follow. Bergen's
noon readings climb from 0.9 degrees on 1 January to 19.1 on 1 July. The last line asked for 1 July
twice and then 1 January, and got all three, in that order.

### Sorting with np.argsort

`np.argsort` returns the positions of the values in sorted order. Indexing with them sorts, and the
same positions sort any other array that lines up with the first:

In [4]:
means = np.nanmean(readings, axis=1)
order = np.argsort(means, kind="stable")

print(means.round(2))
print(order)
print(names[order])
print(means[order].round(2))

[ 8.    6.5  -4.48  3.5 ]
[2 3 1 0]
['Svalbard' 'Tromso' 'Oslo' 'Bergen']
[-4.48  3.5   6.5   8.  ]


`order` lists the stations from coldest to warmest, with Svalbard, row 2, first at -4.48 degrees and
Bergen, row 0, last at 8.0. One set of positions sorted the names and the means, so the two stay
matched. `kind="stable"` keeps equal values in the order they had in the array. The default kind is
not stable, and NumPy's documentation makes no promise about the order it gives equal values.
Readings have one decimal place, so many of them are equal, and every sort in this notebook asks for
the stable kind.

### The warmest days

`days` holds Bergen's 365 daily means, made with the reshape the **Shape** notebook taught. The last
five positions of its sort are the five warmest days, and `[::-1]` puts the warmest first:

In [5]:
days = readings[0].reshape(365, 24).mean(axis=1)
warmest = np.argsort(days, kind="stable")[-5:][::-1]

print(warmest)
print(days[warmest].round(2))
for day in warmest:
    print((START + timedelta(days=int(day))).date())

[203 195 198 199 202]
[17.05 17.04 17.04 17.01 17.01]
2025-07-23
2025-07-15
2025-07-18
2025-07-19
2025-07-22


A position in `days` is a day of the year, so `START` plus that many days is its date, and all five
fell between 15 and 23 July. `int(day)` turns NumPy's integer into Python's, as the **Why NumPy**
notebook did, because `timedelta` takes only Python's.

### Pairing a list of rows with a list of hours

`np.nanargmin` gives the position of each row's smallest value, leaving out `nan`. With one list of
rows and one of hours, NumPy pairs them, the first row with the first hour and so on:

In [6]:
coldest = np.nanargmin(readings, axis=1)

print(coldest)
print(readings[np.arange(4), coldest])
print(np.nanmin(readings, axis=1))
for name, hour in zip(names, coldest):
    print(f"{name:8} {START + timedelta(hours=int(hour))}")

[459 363 267 171]
[ -4.8  -6.3 -17.3  -9.2]
[ -4.8  -6.3 -17.3  -9.2]
Bergen   2025-01-20 03:00:00
Oslo     2025-01-16 03:00:00
Svalbard 2025-01-12 03:00:00
Tromso   2025-01-08 03:00:00


Four rows and four hours made four readings, each station's coldest, the same values `np.nanmin`
gives. Every one fell at 03:00 in January, on a different day at each station.

### A grid with np.ix_

For every row with every position, a grid, the two lists must not pair up. `np.ix_` turns them into
a column of rows and a row of positions:

In [7]:
selector = np.ix_([0, 3], noons)
grid = readings[selector]

print(selector[0].shape, selector[1].shape)
print(grid.shape)
print(grid)

(2, 1) (1, 12)
(2, 12)
[[ 0.9  1.3  2.7  7.2 11.9 16.6 19.1 18.1 16.3 12.5  8.3  4.3]
 [-2.2 -3.5 -0.4  2.4  7.1 11.8 14.3 15.  11.5  7.7  3.5 -0.5]]


Bergen and Tromso at all twelve noons came back as two rows of twelve. The shapes (2, 1) and (1, 12)
combine into (2, 12) by broadcasting, which the **Broadcasting** notebook explains. The Common
errors below show what happens when the two lists go in as they are.

### A fancy index returns a copy

`readings[[0, 1]]` and the slice `readings[0:2]` pick the same two rows. `np.shares_memory` says
whether two arrays use the same memory:

In [8]:
picked = readings[[0, 1]]
sliced = readings[0:2]

print(np.array_equal(picked, sliced))
print(np.shares_memory(sliced, readings))
print(np.shares_memory(picked, readings))
print(np.shares_memory(readings[readings > 18], readings))

True
True
False
False


The slice is a view of `readings`. The list of positions made a copy, and so did the mask from the
**Boolean Masks** notebook. A change to the copy stays in the copy:

In [9]:
picked[0, 0] = 99.0

print(picked[0, 0], readings[0, 0])

99.0 -3.6


### Assigning through a fancy index

With the positions on the left of `=`, NumPy writes into the array itself. `scratch` is a copy kept
for the purpose, so that `readings` stays as it was:

In [10]:
scratch = readings.copy()
scratch[[0, 2], 0] = 0.0

print(readings[:, 0])
print(scratch[:, 0])

[ -3.6  -3.5 -14.6  -6.7]
[ 0.  -3.5  0.  -6.7]


The first hour at Bergen and Svalbard is now 0.0 in `scratch`. A copy is made only when a fancy
index is read.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/07-fancy-indexing-solutions.ipynb).

**1.** Pick Oslo's and Tromso's rows, 1 and 3, with one index, and print the shape of the result.

In [11]:
# your code here

**2.** Print Bergen's readings at midnight on the first of every month, using `hour_of_year` and a list of positions.

In [12]:
# your code here

**3.** Print the station names from the warmest yearly mean to the coldest, using `np.argsort` with `kind="stable"`.

In [13]:
# your code here

**4.** Find Oslo's three warmest hours with `np.argsort`, warmest first, and print the date and time of each with `START` and `timedelta`.

In [14]:
# your code here

**5.** Make a grid of Svalbard and Oslo, rows 2 and 1, at noon on 1 January and noon on 1 July with `np.ix_`, and print it.

In [15]:
# your code here

## Common errors

### IndexError: arrays used as indices must be of integer (or boolean) type

In [16]:
spread = np.linspace(0, HOURS - 1, 5)
print(spread)

readings[0, spread]

[   0.   2189.75 4379.5  6569.25 8759.  ]


IndexError: arrays used as indices must be of integer (or boolean) type

`np.linspace` returns decimals, and here the second position is 2189.75. A position has to be a
whole number, so NumPy raises instead of rounding. Ask `linspace` for integers with `dtype=int`,
which drops the fraction, or convert the positions with `.astype(int)`:

In [17]:
spread = np.linspace(0, HOURS - 1, 5, dtype=int)

print(spread)
print(readings[0, spread])

[   0 2189 4379 6569 8759]
[-3.6  2.5 18.6 12.8 -1.7]


### IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (2,) (12,)

In [18]:
readings[[0, 3], noons]

IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (2,) (12,) 

Two lists of positions pair up, the first row with the first noon and so on, so they have to be the
same length, and here one is 2 long and the other 12. The cell meant a grid, every station with
every noon, which is what `np.ix_` makes: `readings[np.ix_([0, 3], noons)]`. Two lists of the same
length raise nothing, and give pairs, as each station's coldest hour did above.

### No error, and a tuple that picked one reading

In [19]:
print(readings[(0, 2)])
print(readings[0, 2])
print(readings[[0, 2]].shape)

-3.7
-3.7
(2, 8760)


Python hands `readings[0, 2]` to NumPy as the tuple `(0, 2)`, so the two are the same index, row 0
at position 2, which is Bergen at 02:00 on 1 January. A list is what NumPy reads as several positions
along one dimension. Write positions in square brackets, or as an array, whenever they mean rows to
pick.

### No error, and a change made to a copy

In [20]:
scratch = readings.copy()
scratch[[0, 2]][:, 0] = 0.0

print(scratch[:, 0])

[ -3.6  -3.5 -14.6  -6.7]


The first index, `scratch[[0, 2]]`, read two rows into a new array, and `[:, 0] = 0.0` wrote into
that new array, which nothing keeps, so `scratch` still holds the original readings. Put every index
in one pair of brackets, `scratch[[0, 2], 0] = 0.0`, so that NumPy sees the whole assignment at once
and writes into `scratch`.

### No error, and a repeated position counted once

`months` gives the month of every hour, 0 for January, made by `np.repeat`, which repeats each
month's number once for each of its hours. A tally of Bergen's frost hours by month adds 1 at the
month of each frost hour:

In [21]:
DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
months = np.repeat(np.arange(12), [24 * length for length in DAYS_IN_MONTH])
frost_months = months[readings[0] < 0]

tally = np.zeros(12, dtype=int)
tally[frost_months] += 1

print(frost_months.size)
print(tally, tally.sum())

1220
[1 1 1 0 0 0 0 0 0 0 1 1] 5


Bergen had 1,220 frost hours, and the tally adds up to 5. `tally[frost_months] += 1` reads the tally
at every position, adds 1 to each value read, and writes the results back. January appears 441 times
among the positions, and each of those writes puts 1 in the same place. `np.add.at` adds once for
every appearance of a position, and `np.bincount` counts the appearances directly:

In [22]:
tally = np.zeros(12, dtype=int)
np.add.at(tally, frost_months, 1)

print(tally, tally.sum())
print(np.bincount(frost_months, minlength=12))

[441 323  86   0   0   0   0   0   0   0  40 330] 1220
[441 323  86   0   0   0   0   0   0   0  40 330]


### No error, and the missing hours sorted as the warmest

In [23]:
svalbard_order = np.argsort(readings[2], kind="stable")

print(svalbard_order[-3:])
print(readings[2, svalbard_order[-3:]])
print(START + timedelta(hours=int(svalbard_order[-1])))

[1461 1462 1463]
[nan nan nan]
2025-03-02 23:00:00


`np.argsort` puts `nan` after every number, so the last positions of Svalbard's sort are its missing
hours on 2 March, and taking the last few as the warmest gives readings that do not exist. Leave the
missing hours out of the positions first, with a mask:

In [24]:
svalbard_order = svalbard_order[~np.isnan(readings[2, svalbard_order])]

print(svalbard_order.size)
print(readings[2, svalbard_order[-3:]])
print(START + timedelta(hours=int(svalbard_order[-1])))

8736
[8.2 8.2 8.3]
2025-07-13 15:00:00


8,736 positions remain, and Svalbard's warmest hour was 8.3 degrees, at 15:00 on 13 July.

## Recap

- Pick rows and hours with a list of positions, in any order and with repeats
- Sort with `np.argsort(..., kind="stable")`, and use the positions to sort or label other arrays
- Pair two lists of positions, or take a grid of them with `np.ix_`
- Expect a copy from a fancy index or a mask, and check with `np.shares_memory`
- Write through a fancy index in one pair of brackets, and count repeated positions with `np.add.at`

## What is next

The **Broadcasting** notebook explains how the shapes (2, 1) and (1, 12) from `np.ix_` combined into
a grid, and how to read the error when two shapes cannot combine.

---

&#8592; **Previous:** [Boolean Masks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/06-boolean-masks.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/08-broadcasting.ipynb) &#8594;
